# MuleNet-Z — Enclave V2 Privacy Demo

**Purpose:** Demonstrate application-level privacy controls for cross-bank
transaction-risk analysis on IBM LinuxONE.

This prototype demonstrates encrypted bank submissions, bank-specific
encryption keys, a simulated attestation gate, encrypted bank-specific
alerts, and isolation checks.

**Important:** Attestation and enclave execution are software simulations.
This notebook does not establish hardware-backed Secure Execution or
production remote attestation.


## 1. Configure the demo

Use the sample dataset and one day of transactions to limit memory use.
The current fallback model is the combined XGBoost score file.


In [ ]:
import sys
import os
from pathlib import Path
ROOT = Path.home() / 'shared'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pathlib import Path
import os
import time
import pandas as pd

from mulenet.enclave import (
    INSTS,
    make_keyring,
    bank_submit,
    measure_enclave,
    release_keys,
    enclave_run,
    verify_isolation,
    read_my_alerts,
)

ROOT = Path.home() / "shared"
SCORES = ROOT / "results/scores_sample/xgb_combined.csv"
OUT = ROOT / "results/alerts_enc"

assert SCORES.exists(), f"Score file not found: {SCORES}"

print("Project:", ROOT)
print("Score source:", SCORES.name)
print("Banks:", list(INSTS))
print("Sample mode: enabled")
print("Transaction window: 1 day")


## 2. Create bank-specific keys and encrypted submissions

The keyring is local and must never be committed to Git or shared publicly.
Only print its path and permissions—not the key contents.


In [ ]:
ring = make_keyring()

keyring_path = ROOT / "secrets/keyring.json"
if keyring_path.exists():
    print("Keyring exists:", keyring_path)
    print("Permissions:", oct(keyring_path.stat().st_mode & 0o777))

start = time.perf_counter()
subs = {
    bank: bank_submit(bank, ring, sample=True, days=1)
    for bank in INSTS
}
submission_seconds = time.perf_counter() - start

print("Encrypted submissions created for:", list(subs.keys()))
print(f"Submission preparation time: {submission_seconds:.2f} seconds")


## 3. Simulated attestation gate

The enclave measurement is a software-computed fingerprint. A deliberately
incorrect expected measurement should be rejected before keys are released.


In [ ]:
measurement = measure_enclave()
print("Measured fingerprint:", measurement)

try:
    release_keys(ring, measurement, expected="0" * 64)
    raise AssertionError("Incorrect measurement was unexpectedly accepted")
except PermissionError as exc:
    print("Tampered measurement rejected:", exc)

keys = release_keys(ring, measurement, expected=measurement)
print("Matching measurement accepted; keys released for the demo.")


## 4. Run Enclave V2

The pipeline decrypts bank submissions inside the software prototype,
joins the sample transaction data, applies combined model scores, and
writes separate encrypted alert files for each bank.


In [ ]:
start = time.perf_counter()

result = enclave_run(
    subs,
    keys,
    scores_csv=str(SCORES),
    out_dir=OUT,
    hmac_key=ring["hmac"],
)

enclave_runtime = time.perf_counter() - start

print("Pipeline result:")
print(result)
print(f"Measured enclave_run wall time: {enclave_runtime:.2f} seconds")
print("Encrypted alert files:")
for bank in INSTS:
    alert_file = OUT / f"{bank}_alerts.enc"
    print(f"{bank}: exists={alert_file.exists()}")


## 5. Verify bank isolation

Each bank should be able to read its own alert file. The isolation check
should report that the alerts map only to that bank's accounts and that
other banks' keys cannot decrypt its file.


In [ ]:
report = verify_isolation(OUT, ring, sample=True)
print("Isolation report:")
print(report)


## 6. Preview one bank's own alerts

This preview uses the B1 bank-specific read function. Raw account identifiers
are not included in the displayed preview; the bank-specific pseudonym and
risk evidence are shown instead.


In [ ]:
preview = read_my_alerts("B1", ring, OUT, sample=True)

safe_columns = [
    col for col in [
        "account_pseudonym",
        "score",
        "rank",
        "evidence.amount_in",
        "evidence.amount_out",
        "evidence.pass_through",
        "evidence.senders_from_members",
        "evidence.receivers_at_members",
    ]
    if col in preview.columns
]

print("B1 alert count:", len(preview))
display(preview[safe_columns].head(10))


## 7. Demo summary and limitations

### Demonstrated
- Bank-specific encrypted submissions and encrypted alert files.
- A separate bank key design and stable consortium pseudonyms.
- Rejection of an incorrect software-computed measurement.
- Bank-level alert isolation checks.
- Risk ranking from the combined XGBoost score file.

### Limitations
- The enclave and attestation mechanism are simulated in software.
- This is not proof of IBM Secure Execution hardware isolation or
  production remote attestation.
- This run uses the combined XGBoost scores, not GNN scores.
- Alerts are selected using the implementation's top-1% per-bank
  test-split budget; they are risk candidates, not confirmed fraud.
- Runtime figures describe the measured operations only, not the entire
  end-to-end project workflow.
